# TASK DESCRIPTION

**Legend:**

Young Alex has a beloved BERT model that he carries everywhere on his trusty flash drive. One day, during an excursion along the River Styx, a few drops of water landed on the precious device, corrupting the model's weights.

Heartbroken, Alex rushed home to fix the neural network. After quick analysis, he discovered only the token embeddings were damaged - the rest of the architecture (attention blocks and heads) remained perfectly intact. Now he needs to restore the model's performance on Sentiment Analysis Task.

**Task:**

You need to fix the broken vectors of the Embeddings matrix of the model so as to improve the quality of the model on the task of text sentiment analysis.

**Restrictions:**

- You can not use any other transformer based pre-trained models and LLMs.

- You can not any additional data

- You can not fine-tune or pre-train model

===

When you make a submit, make a Quick Save of the notebook, otherwise we may reject your solution.

You must solve this task on KAGGLE (YOU CAN'T USE CLOUD.RU)

==========

**Легенда:**

Young Alex имеет любимую модель BERT, которую он везде носит на своей надежной флешке. Однажды, во время экскурсии вдоль реки Стикс, несколько капель воды попало на драгоценное устройство, повредив веса модели.

С разбитым сердцем Алекс поспешил домой, чтобы починить нейронную сеть. После быстрого анализа он обнаружил, что повреждены только эмбеддинги токенов — остальная архитектура (блоки внимания и головы) осталась полностью нетронутой.

Теперь ему нужно восстановить производительность модели, оставив все остальные веса замороженными (никакие изменения в механизмах внимания или других компонентах не допускаются). Ваша задача — помочь Алексу достичь этой цели, не нарушив его ностальгическую привязанность к оригинальной модели.

**Задача:**

Вам необходимо починить сломанные вектора матрицы Embeddings модели так, чтобы улучшить качество модели на задаче анализа тональности текста.

**Ограничения:**

- Вы не можете использовать никакие другие предобученные модели на основе архитектуры Трансформер и LLM.

- Вы не можете использовать никакие дополнительные данные.

- Вы не можете дообучать или предобучать модель.

===

При отправке решения сделайте Quick Save ноутбука, иначе мы можем отклонить ваше решение.

Эту задачу необходимо решить на KAGGLE (ВЫ НЕ МОЖЕТЕ ИСПОЛЬЗОВАТЬ CLOUD.RU)


# DEPENDINGS

In [1]:
import numpy as np
import pandas as pd
import torch
np.random.seed(21)

# LOAD DATASET

In [2]:
val_data_path = "./neoai-2025-broken-bert/val_dataset.csv"
test_data_path = "./neoai-2025-broken-bert/test.csv"

temp_df = pd.read_csv(val_data_path)
test_df = pd.read_csv(test_data_path)
temp_df.head(5)

,id,text,labels
0,0,"simple recipe for creamy spaghetti with bacon,...",neutral
1,1,wow heather,neutral
2,2,@Djalfy I sound really Brummie lol but most of...,negative
3,3,the fact my room is so hot is making me feel sick,negative
4,4,just came back from the mall,neutral


In [3]:
len(temp_df)

2500

# LOAD TOKENIZER & MODEL

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("Ilseyar-kfu/broken_bert")

In [5]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# MODEL CHANGES

In [6]:
model = AutoModelForSequenceClassification.from_pretrained("Ilseyar-kfu/broken_bert")

new_embedings = model.bert.embeddings.word_embeddings.weight.detach().numpy().copy()

# There's magic going on here!!! And we get very new !!! new_embedings !!!

model.bert.embeddings.word_embeddings.weight = torch.nn.Parameter(torch.Tensor(new_embedings))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# model = AutoModelForSequenceClassification.from_pretrained('fixed_bert/checkpoint-1278')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
for param in model.parameters():
    param.requires_grad = False

model.bert.embeddings.word_embeddings.weight.requires_grad = True

trainable = [name for name, p in model.named_parameters() if p.requires_grad]
print(trainable)

['bert.embeddings.word_embeddings.weight']


# Training

In [8]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(temp_df, test_size=0.1)

train_encodings = tokenizer(train_df['text'].to_list(), padding=True, truncation=True, max_length=256)
val_encodings = tokenizer(val_df["text"].to_list(), truncation=True, padding=True, max_length=256)

label2id = {'neutral': 0, 'positive': 1, 'negative': 2}
train_labels = [label2id[label] for label in train_df["labels"].to_list()]
val_labels = [label2id[label] for label in val_df["labels"].to_list()]

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)

In [9]:
idx = 0
item = {key: torch.tensor(val[idx]) for key, val in val_encodings.items()}
item

{'input_ids': tensor([  101,  7570, 17417,  2078,  9457, 10733,  1010,  5474,  1998, 11565,
         10733,  1998, 21500, 10610,  2005,  4596,  2007,  9493,  1998, 12323,
          1012,  2204,  2335,   999,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0])}

In [10]:
texts_2_score = val_df["text"].to_list() + test_df["text"].to_list()

In [11]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='./fixed_bert',
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=1e-3,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='loss'
)

trainer = Trainer(model, args, train_dataset=train_dataset, eval_dataset=val_dataset)
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.335805,1.238898
2,1.183682,1.155371
3,1.096942,1.108423
4,1.027977,1.082763
5,0.965574,1.073679
6,0.942985,1.068258
7,0.915355,1.064312
8,0.901571,1.062579
9,0.878961,1.061694
10,0.875714,1.061476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=710, training_loss=1.0124565984161806, metrics={'train_runtime': 146.011, 'train_samples_per_second': 154.098, 'train_steps_per_second': 4.863, 'total_flos': 774694291455000.0, 'train_loss': 1.0124565984161806, 'epoch': 10.0})

# EVALUATION

In [12]:
from sklearn.metrics import f1_score
from numpy import argmax
from transformers import pipeline
# import wandb
# wandb.init(mode= "disabled")

In [13]:
list(val_df["text"])[0]

'Hoisin duck pizza, salt and pepper pizza and gelato for dinner with Edmund and Jade. Good times!'

In [14]:
encoded = tokenizer(list(val_df["text"]), padding=True, truncation=True, max_length=256, return_tensors='pt')
encoded = {k: v.to(model.device) for k, v in encoded.items()}
model(**encoded)

SequenceClassifierOutput(loss=None, logits=tensor([[ 3.6433e-01,  3.1979e-01, -7.7919e-01],
        [ 2.9750e-01, -1.5605e+00,  1.0596e+00],
        [ 5.3907e-01, -1.1001e+00,  4.2352e-01],
        [ 1.8940e-01, -4.1646e-01,  3.4140e-02],
        [-4.6511e-01, -7.2381e-02,  2.4888e-01],
        [ 5.0164e-01, -1.7492e+00,  1.0233e+00],
        [ 8.0335e-01, -1.5787e+00,  4.0676e-01],
        [ 4.3528e-01, -2.3311e-01, -5.0398e-01],
        [-4.9168e-01,  1.9435e+00, -9.6223e-01],
        [ 3.3221e-01, -5.8943e-01, -8.1052e-02],
        [ 2.2743e-01, -2.1493e-01, -1.4060e-01],
        [ 3.0756e-01, -3.6447e-01, -9.5627e-02],
        [ 4.0378e-02,  5.9399e-01, -7.8203e-01],
        [-3.2303e-01, -2.1355e+00,  2.2309e+00],
        [-1.4996e-01,  3.3818e-01, -3.0370e-01],
        [ 1.0256e+00, -1.1072e+00, -3.0304e-01],
        [-2.7718e-02, -3.7555e-01,  2.8391e-01],
        [-2.5635e-01, -6.5043e-02,  3.9560e-02],
        [ 2.1815e-01,  1.5209e-01, -5.1547e-01],
        [ 2.3131e-01, -5.7

In [15]:
from sklearn.metrics import classification_report

def evaluate_on_validation(model, tokenizer, df_val):
    # label_2_dict = {'LABEL_0': 'neutral', "LABEL_1" : 'positive', "LABEL_2": 'negative'}
    # classifier = pipeline("text-classification", model= model, tokenizer = tokenizer)
    # answ = classifier.predict(list(df_val["text"]))
    # answ = [label_2_dict[el["label"]] for el in answ]
    label_2_dict = np.array(['neutral', 'positive', 'negative'])
    encoded = tokenizer(list(val_df["text"]), padding=True, truncation=True, max_length=256, return_tensors='pt')
    encoded = {k: v.to(model.device) for k, v in encoded.items()}
    logits = model(**encoded).logits
    preds = label_2_dict[torch.argmax(logits, dim=1).detach().cpu().numpy()]
    
    # print(f1_score(p.label_ids, preds, average='macro'))
    print(classification_report(df_val["labels"], preds))

In [16]:
evaluate_on_validation(model, tokenizer, val_df)

              precision    recall  f1-score   support

    negative       0.60      0.47      0.53        85
     neutral       0.40      0.56      0.47        77
    positive       0.53      0.45      0.49        88

    accuracy                           0.49       250
   macro avg       0.51      0.49      0.49       250
weighted avg       0.51      0.49      0.49       250



# MODEL SCORING
When you make a submit, 
1. Make a Quick Save of the notebook, otherwise we may reject your solution! 
2. Add notebook version to the comment for the submit.

===

При отправке решения:

1. Сделайте Quick Save ноутбука, иначе мы можем отклонить ваше решение!
2. Добавьте версию ноутбука в комментарий к отправке.

In [12]:
import hashlib

def create_submission(model, tokenizer, df_test):
    label_2_dict = {'LABEL_0': 'neutral', "LABEL_1" : 'positive', "LABEL_2": 'negative'}
    classifier = pipeline("text-classification", model= model, tokenizer = tokenizer)
    answ = classifier.predict(list(df_test["text"]))
    answ = [label_2_dict[el["label"]] for el in answ]
    
    df = pd.DataFrame({"labels" : answ, "id": df_test['id']})
    hsh = hashlib.sha256(df.to_csv(index=False).encode('utf-8')).hexdigest()[:8]
    submit_path = f"submit_{hsh}.csv"
    print(f"SUBMIT_NAME: {submit_path}")
    print(df.head(10))
    df.to_csv(submit_path,index=False)

In [ ]:
create_submission(model, tokenizer, test_df)